# Telecom Customer Churn Prediction
**Module:** CSC-44112 — Advanced Applications of AI and ML | Assessment Part 2
**Dataset:** IBM Watson Analytics – Telco Customer Churn ([Kaggle](https://www.kaggle.com/datasets/blastchar/telco-customer-churn))
**Task:** Binary classification — predict whether a customer will churn (Yes / No)

---
| Notebook Section | Aligned to Report Section | Mark Weight |
|---|---|---|
| Section 2 — EDA, Data Cleaning & Preprocessing | **Section 3 — Exploratory Data Analysis** | 20% |
| Section 3 — Model Development & Hyperparameter Tuning | **Section 4 — Methodology** | 25% |
| Section 4 — Results & Evaluation | **Section 5 — Results and Evaluation** | 20% |


## Section 1 — Imports and Setup
*Aligned to: All report sections*

In [ ]:
# ── Section 1: Imports and Setup ─────────────────────────────────────────
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid')

from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import (train_test_split, GridSearchCV,
    StratifiedKFold, cross_val_score, learning_curve)
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, classification_report,
    RocCurveDisplay, ConfusionMatrixDisplay)

print("Libraries loaded successfully.")


## Section 2 — Exploratory Data Analysis, Data Cleaning and Preprocessing
*Aligned to: **Report Section 3 — Exploratory Data Analysis** (600 words, 20%)*
*ILO1: Apply key concepts of EDA to prepare and analyse datasets for machine learning.*

In [ ]:
# ── Section 2.1: Load Dataset and Initial Inspection ─────────────────────
# In Google Colab: upload the CSV file when prompted
from google.colab import files
uploaded = files.upload()

df = pd.read_csv('Telco-Customer-Churn.csv')
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')

print(f"Dataset shape : {df.shape[0]:,} rows × {df.shape[1]} columns")
print(f"\nColumn types:\n{df.dtypes.to_string()}")
print(f"\nFirst 5 rows:")
df.head()


In [ ]:
# ── Section 2.2: Descriptive Statistics ──────────────────────────────────
print("Numerical feature summary:")
print(df[['tenure', 'MonthlyCharges', 'TotalCharges']].describe().round(2))

print("\nCategorical feature counts (selected):")
for col in ['Contract', 'InternetService', 'PaymentMethod']:
    print(f"\n{col}:\n{df[col].value_counts().to_string()}")


In [ ]:
# ── Section 2.3: Missing Value Analysis ──────────────────────────────────
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)
miss_df = pd.DataFrame({'Missing Count': missing, 'Missing %': missing_pct})

print("Missing value audit:")
print(miss_df[miss_df['Missing Count'] > 0])
print(f"\nTotal missing cells : {df.isnull().sum().sum()}")
print("Note: 11 blank strings in TotalCharges coerced to NaN (new customers, tenure ~0).")

# Verify which rows are missing
print("\nSample rows with missing TotalCharges:")
print(df[df['TotalCharges'].isnull()][['tenure','MonthlyCharges','TotalCharges','Churn']].head())


In [ ]:
# ── Section 2.4: Target Variable Distribution ─────────────────────────────
cc = df['Churn'].value_counts()
print(f"Churn = Yes : {cc['Yes']:,}  ({cc['Yes']/len(df)*100:.1f}%)")
print(f"Churn = No  : {cc['No']:,}  ({cc['No']/len(df)*100:.1f}%)")
print(f"Imbalance ratio (No:Yes) = {cc['No']/cc['Yes']:.2f}:1")
print("\nImplication: ROC-AUC and F1 used as primary metrics (not raw accuracy).")


In [ ]:
# ── Section 2.5: Data Cleaning Pipeline ──────────────────────────────────
df_clean = df.copy()

# Step 1: Remove non-informative identifier
df_clean = df_clean.drop(columns=['customerID'])
print("Step 1 — Dropped customerID (non-informative identifier).")

# Step 2: Impute missing TotalCharges with column median
median_tc = df_clean['TotalCharges'].median()
df_clean['TotalCharges'] = df_clean['TotalCharges'].fillna(median_tc)
print(f"Step 2 — Imputed 11 missing TotalCharges values with median (${median_tc:.2f}).")

# Step 3: Normalise SeniorCitizen (0/1 integer → No/Yes string)
df_clean['SeniorCitizen'] = df_clean['SeniorCitizen'].map({0: 'No', 1: 'Yes'})
print("Step 3 — Mapped SeniorCitizen 0/1 → No/Yes for consistent encoding.")

print(f"\nMissing values after cleaning : {df_clean.isnull().sum().sum()}")
print(f"Dataset shape after cleaning  : {df_clean.shape}")


In [ ]:
# ── Section 2.6: Feature Engineering ─────────────────────────────────────
# AvgMonthlySpend: normalises TotalCharges by tenure
# Captures true average monthly spend regardless of contract age
df_clean['AvgMonthlySpend'] = df_clean['TotalCharges'] / (df_clean['tenure'] + 1)

# CustomerSegment: loyalty tier from tenure quartile bands
df_clean['CustomerSegment'] = pd.cut(
    df_clean['tenure'],
    bins=[0, 12, 36, 72],
    labels=['New (0-12m)', 'Mid-term (13-36m)', 'Loyal (37-72m)'],
    include_lowest=True
).astype(str)

print("Feature engineering complete. New features:")
print(df_clean[['tenure','TotalCharges','AvgMonthlySpend','CustomerSegment']].head(8))
print(f"\nCustomerSegment distribution:")
print(df_clean['CustomerSegment'].value_counts())


In [ ]:
# ── Section 2.7: Encoding and Train/Test Split ────────────────────────────
# Encode target variable
df_clean['Churn'] = df_clean['Churn'].map({'Yes': 1, 'No': 0})

# One-hot encode all categorical features
cat_cols = df_clean.select_dtypes(include='object').columns.tolist() + ['CustomerSegment']
df_model = pd.get_dummies(df_clean, columns=cat_cols, drop_first=False)
df_model = df_model.fillna(df_model.median(numeric_only=True))

X = df_model.drop(columns=['Churn'])
y = df_model['Churn']

# 80/20 stratified split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y)

# StandardScaler for Logistic Regression (fitted on train only — no data leakage)
scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)

print(f"Features after one-hot encoding : {X.shape[1]}")
print(f"Training set                    : {X_train.shape[0]:,} samples (80%)")
print(f"Test set                        : {X_test.shape[0]:,} samples (20%)")
print(f"Churn rate — train : {y_train.mean()*100:.1f}%  |  test : {y_test.mean()*100:.1f}%")
print("\nPreprocessing complete. StandardScaler fitted on training data only.")


In [ ]:
# ── Section 2.8: EDA Visualisations — Figure 1 ───────────────────────────
# Six-panel figure covering churn rate, contract, internet service,
# tenure distribution, monthly charges distribution, scatter plot
colors = ['#4D3425', '#E4512B']
fig, axes = plt.subplots(2, 3, figsize=(16, 10))

# 1. Churn rate pie
axes[0,0].pie(cc.values, labels=cc.index, autopct='%1.1f%%', colors=colors,
              startangle=90, explode=(0.05, 0.05), textprops={'fontsize': 12})
axes[0,0].set_title('Churn Rate Distribution')

# 2. Contract type vs churn
ct = df.groupby(['Contract','Churn']).size().unstack(fill_value=0)
(ct.div(ct.sum(axis=1),axis=0)*100).plot(kind='bar', stacked=True,
    ax=axes[0,1], color=colors, rot=10, edgecolor='white')
axes[0,1].yaxis.set_major_formatter(mtick.PercentFormatter())
axes[0,1].set_title('Churn Rate by Contract Type')
axes[0,1].set_ylabel('% Customers'); axes[0,1].legend(title='Churn'); axes[0,1].set_xlabel('')

# 3. Internet service vs churn
is_ct = df.groupby(['InternetService','Churn']).size().unstack(fill_value=0)
(is_ct.div(is_ct.sum(axis=1),axis=0)*100).plot(kind='bar', stacked=True,
    ax=axes[0,2], color=colors, rot=0, edgecolor='white')
axes[0,2].yaxis.set_major_formatter(mtick.PercentFormatter())
axes[0,2].set_title('Churn Rate by Internet Service')
axes[0,2].set_ylabel('% Customers'); axes[0,2].legend(title='Churn'); axes[0,2].set_xlabel('')

# 4. Tenure distribution
for label, col in [('No','#4D3425'), ('Yes','#E4512B')]:
    axes[1,0].hist(df[df['Churn']==label]['tenure'], bins=30, alpha=0.65,
                   color=col, label=f'Churn={label}', edgecolor='white')
axes[1,0].set_title('Tenure Distribution by Churn')
axes[1,0].set_xlabel('Tenure (months)'); axes[1,0].legend()

# 5. Monthly charges distribution
for label, col in [('No','#4D3425'), ('Yes','#E4512B')]:
    axes[1,1].hist(df[df['Churn']==label]['MonthlyCharges'], bins=30, alpha=0.65,
                   color=col, label=f'Churn={label}', edgecolor='white')
axes[1,1].set_title('Monthly Charges by Churn')
axes[1,1].set_xlabel('Monthly Charges ($)'); axes[1,1].legend()

# 6. Scatter: tenure vs monthly charges
for label, col in [('No','#4D3425'), ('Yes','#E4512B')]:
    s = df[df['Churn']==label]
    axes[1,2].scatter(s['tenure'], s['MonthlyCharges'], alpha=0.3,
                      s=10, color=col, label=f'Churn={label}')
axes[1,2].set_title('Tenure vs Monthly Charges')
axes[1,2].set_xlabel('Tenure (months)'); axes[1,2].set_ylabel('Monthly Charges ($)')
axes[1,2].legend()

plt.suptitle('Figure 1 — EDA: Key Patterns in Telco Customer Churn',
             fontsize=15, fontweight='bold')
plt.tight_layout()
plt.savefig('fig1_eda.png', dpi=120, bbox_inches='tight')
plt.show()


In [ ]:
# ── Section 2.9: Feature Correlation with Churn — Figure 2 ───────────────
df_enc = df.copy()
le = LabelEncoder()
for col in df_enc.select_dtypes(include='object').columns:
    df_enc[col] = le.fit_transform(df_enc[col].astype(str))
df_enc['TotalCharges'] = df_enc['TotalCharges'].fillna(df_enc['TotalCharges'].median())

churn_corr = df_enc.corr()['Churn'].drop('Churn').sort_values(ascending=False)

plt.figure(figsize=(13, 5))
plt.bar(churn_corr.index, churn_corr.values,
        color=['#E4512B' if v > 0 else '#4D3425' for v in churn_corr.values],
        edgecolor='white')
plt.axhline(0, color='black', lw=0.8)
plt.xticks(rotation=45, ha='right', fontsize=9)
plt.ylabel('Pearson Correlation with Churn')
plt.title('Figure 2 — Feature Correlation with Churn Target', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('fig2_correlation.png', dpi=120, bbox_inches='tight')
plt.show()

print("Top 5 positive correlates (churn risk factors):")
print(churn_corr[churn_corr > 0].head(5).round(3))
print("\nTop 5 negative correlates (retention factors):")
print(churn_corr[churn_corr < 0].head(5).round(3))


## Section 3 — Model Development and Hyperparameter Tuning
*Aligned to: **Report Section 4 — Methodology** (800 words, 25%)*
*ILO2: Assess suitability of ML algorithms, considering efficiency and interpretability.*

| Model | Rationale | Category |
|---|---|---|
| Logistic Regression | Probabilistic linear baseline; interpretable log-odds coefficients | Parametric |
| Decision Tree | Captures non-linear splits; human-readable rules; tree baseline | Tree |
| Random Forest (tuned) | Bagging ensemble; reduces variance; robust to noisy features | Ensemble |
| Gradient Boosting (tuned) | Histogram-based boosting; highest tabular accuracy; fast training | Ensemble |


In [ ]:
# ── Section 3.1: Cross-Validation Setup ──────────────────────────────────
# 5-fold Stratified K-Fold: preserves 26.5% churn ratio in every fold
cv5 = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
print("Cross-validation strategy : 5-Fold Stratified K-Fold")
print("Primary scoring metric    : ROC-AUC")
print("Random state              : 42 (reproducibility)")


In [ ]:
# ── Section 3.2: Hyperparameter Tuning — Random Forest ───────────────────
print("Tuning Random Forest via GridSearchCV...")
rf_grid = GridSearchCV(
    RandomForestClassifier(random_state=42),
    {'n_estimators':   [100, 200],       # Number of trees
     'max_depth':      [None, 10, 20],   # Max depth per tree
     'min_samples_leaf': [1, 2],         # Min samples at leaf node
     'max_features':   ['sqrt', 'log2']},# Features considered per split
    cv=cv5, scoring='roc_auc', n_jobs=-1  # 48 combos × 5 folds = 240 fits
)
rf_grid.fit(X_train, y_train)
print(f"  Best parameters : {rf_grid.best_params_}")
print(f"  Best CV ROC-AUC : {rf_grid.best_score_:.4f}")


In [ ]:
# ── Section 3.3: Hyperparameter Tuning — Gradient Boosting ───────────────
print("Tuning Gradient Boosting via GridSearchCV...")
hgb_grid = GridSearchCV(
    HistGradientBoostingClassifier(random_state=42),
    {'max_iter':          [100, 200],       # Boosting rounds
     'learning_rate':     [0.05, 0.1, 0.2], # Shrinkage rate
     'max_depth':         [3, 4, 5],         # Tree depth
     'l2_regularization': [0.0, 1.0]},       # L2 penalty on leaves
    cv=cv5, scoring='roc_auc', n_jobs=-1    # 36 combos × 5 folds = 180 fits
)
hgb_grid.fit(X_train, y_train)
print(f"  Best parameters : {hgb_grid.best_params_}")
print(f"  Best CV ROC-AUC : {hgb_grid.best_score_:.4f}")


## Section 4 — Results and Evaluation
*Aligned to: **Report Section 5 — Results and Evaluation** (700 words, 20%)*
*ILO3: Apply data science tools and libraries for visualisation and ML to solve authentic problems.*

In [ ]:
# ── Section 4.1: Train All Models and Collect Test Set Metrics ────────────
model_defs = {
    'Logistic Regression':       (LogisticRegression(max_iter=1000, random_state=42), X_train_sc, X_test_sc),
    'Decision Tree':             (DecisionTreeClassifier(random_state=42),             X_train,    X_test),
    'Random Forest (tuned)':     (rf_grid.best_estimator_,                             X_train,    X_test),
    'Gradient Boosting (tuned)': (hgb_grid.best_estimator_,                            X_train,    X_test),
}

results, trained = [], {}
for name, (mdl, Xtr, Xte) in model_defs.items():
    mdl.fit(Xtr, y_train)
    p    = mdl.predict(Xte)
    prob = mdl.predict_proba(Xte)[:, 1]
    trained[name] = (mdl, p, prob)
    results.append({'Model': name,
        'Accuracy':  round(accuracy_score(y_test, p),  4),
        'Precision': round(precision_score(y_test, p), 4),
        'Recall':    round(recall_score(y_test, p),    4),
        'F1':        round(f1_score(y_test, p),        4),
        'ROC-AUC':   round(roc_auc_score(y_test, prob), 4)})

res_df = pd.DataFrame(results).sort_values('ROC-AUC', ascending=False)
print("Model Performance on Held-out Test Set (n = 1,409):")
print(res_df.to_string(index=False))


In [ ]:
# ── Section 4.2: Confusion Matrices + ROC Curves + Metric Bar — Figure 3 ─
palette = ['#1f77b4','#ff7f0e','#2ca02c','#d62728']
fig, axes = plt.subplots(2, 3, figsize=(18, 11))

cm_axes = [axes[0,0], axes[0,1], axes[0,2], axes[1,0]]
for ax, (name, (_, preds, _)), col in zip(cm_axes, trained.items(), palette):
    ConfusionMatrixDisplay(confusion_matrix(y_test, preds),
        display_labels=['No Churn','Churn']).plot(ax=ax, colorbar=False, cmap='Blues')
    ax.set_title(name, fontsize=11)

ax_roc = axes[1,1]
for (name, (_, _, probs)), col in zip(trained.items(), palette):
    RocCurveDisplay.from_predictions(y_test, probs, name=name, ax=ax_roc, color=col, lw=2)
ax_roc.plot([0,1],[0,1],'k--', lw=1, label='Random (AUC=0.50)')
ax_roc.set_title('ROC Curves', fontsize=12); ax_roc.legend(loc='lower right', fontsize=8)

ax_bar = axes[1,2]; x = np.arange(len(res_df)); w = 0.15
for i, m in enumerate(['Accuracy','Precision','Recall','F1','ROC-AUC']):
    ax_bar.bar(x+i*w, res_df[m], w, label=m,
               color=['#1f77b4','#ff7f0e','#2ca02c','#d62728','#9467bd'][i], edgecolor='white')
ax_bar.set_xticks(x+w*2); ax_bar.set_xticklabels(res_df['Model'], rotation=18, ha='right', fontsize=8)
ax_bar.set_ylim(0.4, 1.0); ax_bar.legend(fontsize=7, loc='lower right')
ax_bar.set_title('Model Metric Comparison', fontsize=12)

plt.suptitle('Figure 3 — Model Evaluation: Confusion Matrices, ROC Curves and Metric Comparison',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('fig3_evaluation.png', dpi=120, bbox_inches='tight')
plt.show()


In [ ]:
# ── Section 4.3: Best Model Classification Report + 5-Fold CV ─────────────
best_name = res_df.iloc[0]['Model']
_, best_preds, best_probs = trained[best_name]

print(f"Best Model: {best_name}")
print("="*55)
print(classification_report(y_test, best_preds, target_names=['No Churn','Churn']))
print(f"ROC-AUC : {roc_auc_score(y_test, best_probs):.4f}")

print("\n5-Fold Stratified CV ROC-AUC (training set only):")
for name, (mdl, _, _) in trained.items():
    Xtr = X_train_sc if name == 'Logistic Regression' else X_train
    sc = cross_val_score(mdl, Xtr, y_train, cv=cv5, scoring='roc_auc', n_jobs=-1)
    print(f"  {name:<35s}  {sc.mean():.4f} ± {sc.std():.4f}")


In [ ]:
# ── Section 4.4: Feature Importance + Learning Curves — Figure 4 ──────────
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

best_rf = rf_grid.best_estimator_
imp = pd.DataFrame({'Feature': X_train.columns,
                    'Importance': best_rf.feature_importances_}
    ).sort_values('Importance', ascending=False).head(15)
axes[0].barh(imp['Feature'][::-1], imp['Importance'][::-1],
             color=sns.color_palette('RdYlGn_r', 15), edgecolor='white')
axes[0].set_xlabel('Feature Importance (Mean Decrease in Impurity)')
axes[0].set_title('Top 15 Feature Importances — Random Forest (Tuned)', fontsize=12)

ts, tr, vr = learning_curve(
    HistGradientBoostingClassifier(**hgb_grid.best_params_, random_state=42),
    X_train, y_train, cv=cv5, scoring='roc_auc',
    train_sizes=np.linspace(0.1, 1.0, 8), n_jobs=-1)
axes[1].plot(ts, tr.mean(1), 'o-', color='#E4512B', lw=2, label='Training')
axes[1].fill_between(ts, tr.mean(1)-tr.std(1), tr.mean(1)+tr.std(1), alpha=0.15, color='#E4512B')
axes[1].plot(ts, vr.mean(1), 'o-', color='#4D3425', lw=2, label='Validation')
axes[1].fill_between(ts, vr.mean(1)-vr.std(1), vr.mean(1)+vr.std(1), alpha=0.15, color='#4D3425')
axes[1].set_xlabel('Training Set Size'); axes[1].set_ylabel('ROC-AUC')
axes[1].set_title('Learning Curves — Gradient Boosting (Tuned)', fontsize=12); axes[1].legend()

plt.suptitle('Figure 4 — Feature Importance and Learning Curves',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('fig4_importance_learning.png', dpi=120, bbox_inches='tight')
plt.show()

print("Top 5 most predictive features:")
print(imp.head(5)[['Feature','Importance']].to_string(index=False))
